<div style="background:linear-gradient(135deg,#7a3d00 0%,#b45309 55%,#d97706 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#ffe9c7;font-weight:700;text-transform:uppercase">Chapter 19 · Solutions</div>
  <div style="font-size:36px;font-weight:900;line-height:1.1;margin:10px 0 6px">Practice Challenges, Worked Answers ✅</div>
  <div style="font-size:15px;color:#ffe6cc;max-width:700px;line-height:1.6">Full solutions to the five "Duplicates & Inconsistencies" challenges. Try them yourself first, then compare.</div>
  <div style="margin-top:16px;font-size:13px;color:#ffe2bf">Statistics, Data Science and AI: A Visual Handbook · John Fisher · 2026</div>
</div>

## ⚙️ Setup

In [1]:
import numpy as np
import pandas as pd
import re
print("Ready.")

Ready.


<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#0891b2;letter-spacing:1px">CHALLENGE 1 · DEDUP ON A KEY</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">One row per customer</div>
<div style="color:#4a5578;margin-top:6px">This table has an exact duplicate row and a customer who appears twice with different spend. Drop the exact duplicate, then keep one row per customer_id (the most recent), and report how many rows you removed.</div>
</div>

In [2]:
df = pd.DataFrame({
    "customer_id":[1,1,2,3,3],
    "name":["Ana","Ana","Ben","Cy","Cy"],
    "spend":[50,50,30,20,75],
})
before = len(df)
no_exact = df.drop_duplicates()
one_per = no_exact.drop_duplicates(subset="customer_id", keep="last")
print(f"start: {before} rows")
print(f"after exact dedup: {len(no_exact)} rows")
print(f"one row per customer (keep last): {len(one_per)} rows")
print(f"removed total: {before - len(one_per)} rows")

start: 5 rows
after exact dedup: 4 rows
one row per customer (keep last): 3 rows
removed total: 2 rows


**Answer:** The identical row drops first (5 to 4 rows), then deduping on **customer_id** with `keep="last"` collapses customer 3's two records to the later one (spend 75), leaving **3 rows**, 2 removed in all. Use `subset=` for a key and `keep=` to choose which record survives; always log the count removed.

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">CHALLENGE 2 · STANDARDIZE A COLUMN</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Collapse the spellings</div>
<div style="color:#4a5578;margin-top:6px">A country column has many spellings of two countries. Standardize it to two canonical values and confirm with value_counts.</div>
</div>

In [3]:
country = pd.Series(["USA","usa","U.S.A."," United States ","UK","uk","U.K.","United States"])
clean = country.str.strip().str.lower().str.replace(".","",regex=False)
mapping = {"usa":"United States","us":"United States","united states":"United States",
           "uk":"United Kingdom","uk ":"United Kingdom","united kingdom":"United Kingdom"}
clean = clean.replace(mapping)
print(clean.value_counts())
print(f"\n{country.nunique()} raw labels -> {clean.nunique()} canonical values")

United States     5
United Kingdom    3
Name: count, dtype: int64

8 raw labels -> 2 canonical values


**Answer:** Strip whitespace, lowercase, drop the periods, then map the variants to canonical names. The seven raw labels collapse to **United States** and **United Kingdom**. The recipe is always discover (value_counts) to define canonical to map (replace) to verify (value_counts again).

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#d97706;letter-spacing:1px">CHALLENGE 3 · COERCE FORMATS</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Text to real types</div>
<div style="color:#4a5578;margin-top:6px">A price column holds values like "1,250", "$90", and "oops". Convert it to numeric, and report how many values could not be parsed.</div>
</div>

In [4]:
price = pd.Series(["1,250","$90","2000","oops","$1,500"])
num = pd.to_numeric(price.str.replace(r"[\$,]","",regex=True), errors="coerce")
print(num)
print(f"\nparsed: {num.notna().sum()}, coerced to NaN: {num.isna().sum()}")

0    1250.0
1      90.0
2    2000.0
3       NaN
4    1500.0
dtype: float64

parsed: 4, coerced to NaN: 1


**Answer:** Strip the `$` and `,` with a regex, then `pd.to_numeric(..., errors="coerce")`. Four values parse; **"oops" becomes NaN** (1 coerced). The key habit: `errors="coerce"` silently creates missing values, so always count `.isna().sum()` afterward so the loss is not invisible. Those NaNs are now a missing-data problem (Chapter 20).

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#059669;letter-spacing:1px">CHALLENGE 4 · NEAR-DUPLICATES</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Score the similarity</div>
<div style="color:#4a5578;margin-top:6px">Using the normalized Levenshtein ratio below, decide which of these company names are likely the same entity (similarity >= 0.85): "Acme Inc", "Acme, Inc.", "Acme Incorporated", "Beta LLC".</div>
</div>

In [5]:
def lev_ratio(a, b):
    m, n = len(a), len(b); d = list(range(n + 1))
    for i in range(1, m + 1):
        prev, d[0] = d[0], i
        for j in range(1, n + 1):
            cur = d[j]; d[j] = min(d[j]+1, d[j-1]+1, prev + (a[i-1]!=b[j-1])); prev = cur
    return 1 - d[n] / max(m, n, 1)

names = ["Acme Inc","Acme, Inc.","Acme Incorporated","Beta LLC"]
norm = [re.sub(r"[^a-z0-9 ]","",s.lower()).strip() for s in names]   # standardize first
for i in range(len(names)):
    for j in range(i+1, len(names)):
        r = lev_ratio(norm[i], norm[j])
        flag = "  <- likely same" if r >= 0.85 else ""
        print(f"{names[i]!r:20} ~ {names[j]!r:20} -> {r:.2f}{flag}")

'Acme Inc'           ~ 'Acme, Inc.'         -> 1.00  <- likely same
'Acme Inc'           ~ 'Acme Incorporated'  -> 0.47
'Acme Inc'           ~ 'Beta LLC'           -> 0.25
'Acme, Inc.'         ~ 'Acme Incorporated'  -> 0.47
'Acme, Inc.'         ~ 'Beta LLC'           -> 0.25
'Acme Incorporated'  ~ 'Beta LLC'           -> 0.12


**Answer:** After normalizing (lowercase, strip punctuation), **"Acme Inc" and "Acme, Inc."** score very high (~0.9+) and are clearly the same company; "Acme Incorporated" is closer to them than to Beta but may fall under a strict 0.85 threshold, which is exactly why you **review** fuzzy matches rather than auto-merge. "Beta LLC" matches nothing. A looser threshold catches more true dupes (recall) but risks false merges (precision).

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#2563eb;letter-spacing:1px">CHALLENGE 5 · WRITE THE RULES</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Validate against a contract</div>
<div style="color:#4a5578;margin-top:6px">Write a validate() that flags: duplicate id, age outside 0-120, status not in {active, inactive}. Run it on the frame below and print every violation.</div>
</div>

In [6]:
df = pd.DataFrame({
    "id":[1,2,2,4],
    "age":[30,-5,45,130],
    "status":["active","inactive","pending","active"],
})
VALID = {"active","inactive"}
def validate(df):
    issues = []
    if not df["id"].is_unique: issues.append(f"duplicate id: {df['id'][df['id'].duplicated()].tolist()}")
    if not df["age"].between(0,120).all(): issues.append(f"age out of range: {df.loc[~df['age'].between(0,120),'age'].tolist()}")
    bad = df.loc[~df["status"].isin(VALID),"status"].tolist()
    if bad: issues.append(f"invalid status: {bad}")
    return issues
v = validate(df)
print("PASS ✅" if not v else f"FAIL ❌: {len(v)} rule(s):")
for x in v: print("   •", x)

FAIL ❌: 3 rule(s):
   • duplicate id: [2]
   • age out of range: [-5, 130]
   • invalid status: ['pending']


**Answer:** The frame fails all three rules: a **duplicate id (2)**, **ages -5 and 130** outside 0-120, and an **invalid status "pending"**. A `validate()` that returns a list of violations turns cleaning into a repeatable, testable contract you run on ingest and again after cleaning. In a real pipeline you would reach for a schema tool like **pandera** or **Great Expectations** to declare these rules.

---
<div style="background:#ffffff;border:1px solid #e6e9f2;border-radius:16px;padding:22px 26px;font-family:Inter,sans-serif">
<div style="font-size:19px;font-weight:800;color:#1a2138">🎉 Nicely done!</div>
<div style="color:#4a5578;line-height:1.8;margin-top:8px">You deduped on a key, collapsed a tangle of spellings, coerced messy text to real types, scored near-duplicates, and codified validation rules. With standardize-then-dedup-then-validate, your data is finally ready to join, group, and model.</div>
</div>

<div style="text-align:center;color:#8b94b3;font-size:12px;margin-top:16px">Statistics, Data Science and AI: A Visual Handbook · © 2026 John Fisher</div>